# Data Understanding

**Objective:** Audit source workbooks, data types, duplicates, and relationships.

The code is split into visible, explainable steps for a demonstration video.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW = ROOT / 'data' / 'raw' / 'Tourism Dataset'
PROCESSED = ROOT / 'data' / 'processed'
RANDOM_STATE = 42
pd.set_option('display.max_columns', 50)

In [ ]:
files = {'transaction': RAW/'Transaction.xlsx', 'user': RAW/'User.xlsx', 'city': RAW/'City.xlsx', 'country': RAW/'Country.xlsx', 'region': RAW/'Region.xlsx', 'continent': RAW/'Continent.xlsx', 'mode': RAW/'Mode.xlsx', 'type': RAW/'Type.xlsx', 'item': RAW/'Item.xlsx', 'updated_item': RAW/'Additional_Data_for_Attraction_Sites'/'Updated_Item.xlsx'}
tables = {name: pd.read_excel(path) for name, path in files.items()}
{name: df.shape for name, df in tables.items()}

In [ ]:
def audit_table(name, df):
    key = df.columns[0]
    return {'dataset': name, 'rows': len(df), 'columns': len(df.columns), 'missing_cells': int(df.isna().sum().sum()), 'duplicate_rows': int(df.duplicated().sum()), 'key_column': key, 'key_is_unique': df[key].is_unique}
audit = pd.DataFrame([audit_table(name, df) for name, df in tables.items()])
audit

In [ ]:
checks = [('Transaction.UserId -> User.UserId', tables['transaction'].UserId, tables['user'].UserId), ('Transaction.AttractionId -> Item.AttractionId', tables['transaction'].AttractionId, tables['item'].AttractionId), ('User.CityId -> City.CityId', tables['user'].CityId, tables['city'].CityId), ('Item.AttractionTypeId -> Type.AttractionTypeId', tables['item'].AttractionTypeId, tables['type'].AttractionTypeId)]
pd.DataFrame([{'relationship': label, 'orphan_rows': int((~left.isin(right)).sum())} for label, left, right in checks])

In [ ]:
item, updated = tables['item'], tables['updated_item']
pd.Series({'same_columns': list(item.columns) == list(updated.columns), 'item_rows': len(item), 'updated_item_rows': len(updated), 'shared_attraction_ids': item.AttractionId.isin(updated.AttractionId).sum()})